## Módulo 3: modelos abiertos con Hugging Face

Corriendo local, en CPU (sin driver NVIDIA activo en esta máquina — más lento que GPU, pero para modelos chicos como este anda bien).

Primer paso: `pipeline` de `transformers`. Es una abstracción que junta 3 cosas que hasta ahora hacíamos por separado o ni veíamos (con Ollama la caja estaba cerrada):
1. **tokenizer** — convierte texto a tokens (números) que entiende el modelo
2. **modelo** — corre inferencia sobre esos tokens
3. **post-proceso** — convierte la salida cruda del modelo en algo legible

Vamos a abrir esa caja: usar `pipeline` para lo rápido, y después separar el tokenizer para ver qué hace por dentro.

In [1]:
from dotenv import load_dotenv
from huggingface_hub import login
import os

load_dotenv()
login(os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### `pipeline`: la forma rápida

Una línea, HF elige tokenizer + modelo + post-proceso según la tarea (`"sentiment-analysis"`).

In [2]:
from transformers import pipeline

clasificador = pipeline("sentiment-analysis")  # sin device=0: no hay GPU, corre en CPU
clasificador("Este curso está siendo durísimo pero vale la pena.")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.6173744797706604}]

### Abriendo la caja: el tokenizer

Mismo concepto que vimos con `tiktoken` en módulo 1 (BPE), pero ahora con el tokenizer real de un modelo abierto (Llama 3.2 — el mismo que corrés local con Ollama, para comparar).

Uso el mirror `NousResearch/Llama-3.2-1B` en vez de `meta-llama/Llama-3.2-1B`: mismo tokenizer, pero sin el paso de aceptar licencia gated en HF (un obstáculo menos para hoy).

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("NousResearch/Llama-3.2-1B")

texto = "Este curso está siendo durísimo pero vale la pena."
tokens = tokenizer.encode(texto)

print(f"Texto: {len(texto)} caracteres")
print(f"Tokens: {len(tokens)}")
for t in tokens:
    print(f"  {t:>6} -> {tokenizer.decode([t])!r}")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Texto: 50 caracteres
Tokens: 13
  128000 -> '<|begin_of_text|>'
   44090 -> 'Este'
   48192 -> ' curso'
   15833 -> ' está'
   82366 -> ' siendo'
   10878 -> ' dur'
   24315 -> 'ís'
   11620 -> 'imo'
   20003 -> ' pero'
   34822 -> ' vale'
    1208 -> ' la'
   95557 -> ' pena'
      13 -> '.'


Fijate cómo palabras con tilde ("está", "durísimo") suelen partirse en más subtokens que sus equivalentes en inglés — el vocabulario BPE de la mayoría de estos modelos se entrena con corpus dominado por inglés. Esto es plata real: el mismo texto en español consume más tokens (y más costo/latencia) que en inglés.